# Shor Bit And Phase Error Correction

Commented revision of `qc-lab6.ipynb`, prepared 2026-11-06. Run cells from top to bottom. Qiskit displays classical bit strings with the highest-index bit on the left. Original import copies have outputs cleared and embedded credentials removed.

Uses nine data qubits and eight syndrome ancillas. Retains the complete conditional correction workflow and removes earlier incomplete experiments and interactive input. The example inserts X on q6 and Z on q2; the code guarantees arbitrary single-qubit error correction, not arbitrary multiple errors. Each parity pair uses low-bit-first decoding in the printed report.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from IPython.display import display
import numpy as np

# 1. Same qubit layout as the Shor circuit above:
# Data blocks: (0, 1, 2), (5, 6, 7), (10, 11, 12)
# Bit syndrome ancillas: 3, 4, 8, 9, 13, 14
# Phase syndrome ancillas: 15, 16
q = QuantumRegister(17, 'q')
c = ClassicalRegister(8, 'c')
qc = QuantumCircuit(q, c)

# 2. Prepare the original state
# psi = [0.6, 0.8]
# psi = [0, 1]
psi = [1/np.sqrt(2), 1/np.sqrt(2)]
qc.initialize(psi, q[0])
qc.barrier()

# 3. Shor encoding
qc.cx(0, 5)
qc.cx(0, 10)
qc.barrier()

qc.h(0)
qc.h(5)
qc.h(10)
qc.barrier()

qc.cx(0, 1)
qc.cx(0, 2)
qc.cx(5, 6)
qc.cx(5, 7)
qc.cx(10, 11)
qc.cx(10, 12)
qc.barrier()

# 4. Introduce errors -- your existing example
# Change/comment these lines to try other data qubits.
qc.x(6)
qc.z(2)
qc.barrier()

# 5. Error detection -- your original H and CX gates
qc.cx(0,3)
qc.cx(1,3)

qc.cx(1,4)
qc.cx(2,4)

qc.cx(5,8)
qc.cx(6,8)

qc.cx(6,9)
qc.cx(7,9)

qc.cx(10,13)
qc.cx(11,13)

qc.cx(11,14)
qc.cx(12,14)

qc.barrier()

qc.h(0)
qc.h(1)
qc.h(2)
qc.h(5)
qc.h(6)
qc.h(7)
qc.h(10)
qc.h(11)
qc.h(12)


qc.barrier()

qc.cx(0,15)
qc.cx(1,15)
qc.cx(2,15)

qc.barrier()

qc.cx(5,15)
qc.cx(6,15)
qc.cx(7,15)

qc.barrier()

qc.cx(5,16)
qc.cx(6,16)
qc.cx(7,16)

qc.barrier()
qc.cx(10,16)
qc.cx(11,16)
qc.cx(12,16)


qc.barrier()

qc.h(0)
qc.h(1)
qc.h(2)
qc.h(5)
qc.h(6)
qc.h(7)
qc.h(10)
qc.h(11)
qc.h(12)

# 6. Measure the syndrome
qc.measure(3, 0)
qc.measure(4, 1)
qc.measure(8, 2)
qc.measure(9, 3)
qc.measure(13, 4)
qc.measure(14, 5)
qc.barrier()

qc.measure(15, 6)
qc.measure(16, 7)
qc.barrier()

print("Shor encoding and error-detection circuit:")
# Circuit is shown after correction and decoding below.

# 7. Correct bit flips using the measured syndrome in each block.
# In low-bit-first order: 10 -> first, 11 -> middle, 01 -> last.
# Check each pair separately so bit and phase errors can coexist.

# Data block [0, 1, 2]
with qc.if_test((c[0], 1)):
    with qc.if_test((c[1], 0)):
        qc.x(0)

with qc.if_test((c[0], 1)):
    with qc.if_test((c[1], 1)):
        qc.x(1)

with qc.if_test((c[0], 0)):
    with qc.if_test((c[1], 1)):
        qc.x(2)

qc.barrier()

# Data block [5, 6, 7]
with qc.if_test((c[2], 1)):
    with qc.if_test((c[3], 0)):
        qc.x(5)

with qc.if_test((c[2], 1)):
    with qc.if_test((c[3], 1)):
        qc.x(6)

with qc.if_test((c[2], 0)):
    with qc.if_test((c[3], 1)):
        qc.x(7)

qc.barrier()

# Data block [10, 11, 12]
with qc.if_test((c[4], 1)):
    with qc.if_test((c[5], 0)):
        qc.x(10)

with qc.if_test((c[4], 1)):
    with qc.if_test((c[5], 1)):
        qc.x(11)

with qc.if_test((c[4], 0)):
    with qc.if_test((c[5], 1)):
        qc.x(12)

qc.barrier()

# 8. Correct a phase flip on one representative qubit of its block.
# c[6], c[7]: 10 -> block 1, 11 -> block 2, 01 -> block 3.
with qc.if_test((c[6], 1)):
    with qc.if_test((c[7], 0)):
        qc.z(0)

with qc.if_test((c[6], 1)):
    with qc.if_test((c[7], 1)):
        qc.z(5)

with qc.if_test((c[6], 0)):
    with qc.if_test((c[7], 1)):
        qc.z(10)

qc.barrier()

# 9. Decode -- reverse the encoding to recover the state on q[0]
qc.cx(0, 2)
qc.cx(0, 1)
qc.cx(5, 7)
qc.cx(5, 6)
qc.cx(10, 12)
qc.cx(10, 11)
qc.barrier()

qc.h(0)
qc.h(5)
qc.h(10)
qc.barrier()

qc.cx(0, 10)
qc.cx(0, 5)
qc.barrier()

print("Complete Shor detection, correction and decoding circuit:")
display(qc.draw('mpl', fold=20))

# 10. Simulate the full circuit, including conditional corrections
qc.save_statevector()
backend = AerSimulator(method='statevector')
result = backend.run(qc, shots=1).result()
final_state = result.get_statevector()
counts = result.get_counts(qc)
print("Syndrome counts:", counts)

# 11. Report detection using your original string and dictionary format
bits = max(counts, key=counts.get).replace(" ", "")[::-1]
bit_qubit = None
error = {"10": 0, "11": 1, "01": 2}

for block, start in enumerate([0, 5, 10], start=1):
    syndrome = bits[(block-1)*2 : block*2]
    if syndrome in error:
        bit_qubit = start + error[syndrome]
        print("Bit flip detected and corrected in block", block, "qubit", bit_qubit)

if bit_qubit is None:
    print("No bit flip error")

phase = bits[6:8]
phase_error = {"10": 1, "11": 2, "01": 3}
phase_block = phase_error.get(phase)

if phase_block is not None:
    print("Phase flip detected and corrected in block", phase_block)
else:
    print("No phase flip error")

# 12. Ancillas retain their measured syndrome after decoding.
# All other data qubits are back in |0>; extract q[0]'s amplitudes.
base = (int(bits[0]) * 2**3 + int(bits[1]) * 2**4
        + int(bits[2]) * 2**8 + int(bits[3]) * 2**9
        + int(bits[4]) * 2**13 + int(bits[5]) * 2**14
        + int(bits[6]) * 2**15 + int(bits[7]) * 2**16)
a = final_state.data[base]
b = final_state.data[base + 1]

print("Recovered output (up to a global phase):")
print(f"({a:.6f})|0> + ({b:.6f})|1>")
print(f"Recovery fidelity: {abs(np.vdot(psi, [a, b]))**2:.6f}")

# Verify recovery up to a physically irrelevant global phase.
assert np.isclose(abs(np.vdot(psi, [a, b]))**2, 1.0)
